# **Load the data**

## 1. Load the dataset

This section imports the data into the notebook. The matching source code is in `src/data_processing.py`.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('../data/raw/RTA Dataset.csv')
df


## 2. Inspect and clean the data

These cells review the dataset, identify missing values, remove selected columns, and drop incomplete rows.

In [ ]:
df.info()

In [ ]:
df.isnull().sum() # Check for missing values

In [ ]:
# Remove columns with many missing values to retain more rows.
columns_to_drop = ['Defect_of_vehicle', 'Service_year_of_vehicle',
    'Fitness_of_casuality', 'Work_of_casuality'
]
df = df.drop(columns=columns_to_drop)

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df

## 4. Save the processed dataset

The cleaned dataset is exported as a CSV file for later use.

In [ ]:
from pathlib import Path

output_path = Path('../data/processed/RTA Dataset cleaned.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

print(f'Saved processed data to: {output_path}')

## 5. Encode categorical variables

Categorical values are converted to numerical labels so the models can use them.

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include=['object', 'category']).columns
label_mappings = {}

for column in categorical_cols:
    encoder = LabelEncoder()
    df[column] = encoder.fit_transform(df[column])
    label_mappings[column] = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))

# Encode text values as numbers, including Accident_severity.
print('Accident severity label mapping:', label_mappings['Accident_severity'])

## 6. Explore relationships and class balance

The following cells visualize feature correlations and inspect the target-class distribution.

In [ ]:
# Calculate the correlation matrix
correlation_matrix = df.corr()

# You can also visualize the correlation matrix using a heatmap
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Variables')
plt.show()


In [ ]:
df['Accident_severity'].value_counts()

# **Random oversampling results**

The experiments use an 80% training split and a 20% test split. The selected features are defined in `SELECTED_FEATURES`.

## 7. Model experiments

The project compares Decision Tree, Random Forest, and XGBoost using multiple resampling approaches and 10-fold validation.

In [ ]:
# prompt: K-Fold
from imblearn.over_sampling import RandomOverSampler
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Use one shared feature set in every model experiment.
SELECTED_FEATURES = [
    'Area_accident_occured', 'Day_of_week', 'Lanes_or_Medians',
    'Road_surface_conditions', 'Age_band_of_driver', 'Light_conditions',
    'Type_of_vehicle', 'Number_of_casualties', 'Cause_of_accident',
    'Number_of_vehicles_involved', 'Age_band_of_casualty',
    'Driving_experience', 'Type_of_collision'
]

n_splits = 10

# Stratification keeps the accident-severity class ratio similar in every fold.
kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

decisiontree

### Random oversampling — Decision Tree

This experiment applies random oversampling to the training portion of each fold before fitting a Decision Tree.

In [ ]:
# Lists to store evaluation metrics for each fold
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

X = df[SELECTED_FEATURES]
y = df['Accident_severity']
# Loop through each fold
for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    print(f"Fold {fold + 1}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Apply RandomOverSampler to the training data of the current fold
    ros = RandomOverSampler(random_state=42)
    X_train_fold, y_train_fold = ros.fit_resample(X_train, y_train)

    # Initialize and train your model (e.g., RandomForestClassifier)
    model = DecisionTreeClassifier() # Or any other classifier
    model.fit(X_train_fold, y_train_fold)

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Store metrics
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    print('classification report:', classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")

# Calculate and print the average metrics across all folds
print("\nAverage Metrics across all folds:")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Average Precision: {np.mean(precision_scores):.4f}")
print(f"Average Recall: {np.mean(recall_scores):.4f}")
print(f"Average F1-score: {np.mean(f1_scores):.4f}")

randomforest

### Random oversampling — Random Forest

This experiment evaluates Random Forest with random oversampling.

In [ ]:
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

X = df[SELECTED_FEATURES]
y = df['Accident_severity']

# Loop through each fold
for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    print(f"Fold {fold + 1}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Apply RandomOverSampler to the training data of the current fold
    ros = RandomOverSampler(random_state=42)
    X_train_fold, y_train_fold = ros.fit_resample(X_train, y_train)

    # Initialize and train your model (e.g., RandomForestClassifier)
    model = RandomForestClassifier() # Or any other classifier
    model.fit(X_train_fold, y_train_fold)

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Store metrics
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    print('classification report:', classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")

# Calculate and print the average metrics across all folds
print("\nAverage Metrics across all folds:")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Average Precision: {np.mean(precision_scores):.4f}")
print(f"Average Recall: {np.mean(recall_scores):.4f}")
print(f"Average F1-score: {np.mean(f1_scores):.4f}")

### Feature importance

This cell ranks input variables according to the fitted Random Forest model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Use the features and target defined in the previous step.
model = RandomForestClassifier(
    n_estimators=300, random_state=42, class_weight='balanced'
)
model.fit(X, y)

# Random Forest provides an importance score for each feature.
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Display the five features that contribute most to the model predictions.
top5 = feature_importance_df.head(5)
top5

XGBoost

### Random oversampling — XGBoost

This experiment evaluates XGBoost with random oversampling.

In [ ]:
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

X = df[SELECTED_FEATURES]
y = df['Accident_severity']


# Loop through each fold
for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    print(f"Fold {fold + 1}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Apply RandomOverSampler to the training data of the current fold
    ros = RandomOverSampler(random_state=42)
    X_train_fold, y_train_fold = ros.fit_resample(X_train, y_train)


    # Initialize and train your model (e.g., RandomForestClassifier)
    model = xgb.XGBClassifier() # Or any other classifier
    model.fit(X_train_fold, y_train_fold)

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Store metrics
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    print('classification report:', classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")

# Calculate and print the average metrics across all folds
print("\nAverage Metrics across all folds:")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Average Precision: {np.mean(precision_scores):.4f}")
print(f"Average Recall: {np.mean(recall_scores):.4f}")
print(f"Average F1-score: {np.mean(f1_scores):.4f}")

# **Random undersampling results**

The experiments use the shared `SELECTED_FEATURES` list and stratified 10-fold cross-validation.

## 8. Random undersampling experiments

This section compares the model approaches under the notebook’s undersampling workflow.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

# Define the number of folds
n_splits = 10  # You can change this value

# Initialize StratifiedKFold
kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

decisiontree

### Random undersampling — Decision Tree

In [ ]:
# Lists to store evaluation metrics for each fold
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
X = df[SELECTED_FEATURES]
y = df['Accident_severity']

# Split the data while preserving the class distribution in each fold.
for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    print(f"Fold {fold + 1}")

    # Select the training and test data for the current fold.
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

    # Initialise and train the baseline model without sampling.
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train_fold, y_train_fold)

    # Predict labels for the current fold test set.
    y_pred = model.predict(X_test_fold)

    # Uncomment these checks to inspect the prediction and label shapes.
    # print(f"Test labels shape: {y_test_fold.shape}")
    # print(f"Predicted labels shape: {y_pred.shape}")

    # Calculate evaluation metrics.
    accuracy = accuracy_score(y_test_fold, y_pred)
    precision = precision_score(y_test_fold, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test_fold, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test_fold, y_pred, average='weighted', zero_division=0)

    # Store the metrics for this fold.
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f}")
    # Uncomment the following line to display a report for each fold.
    # print(classification_report(y_test_fold, y_pred))

# Summarise the average results.
print("\n" + "="*30)
print("AVERAGE METRICS (BASELINE 10-FOLD)")
print("="*30)
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Average Precision: {np.mean(precision_scores):.4f}")
print(f"Average Recall: {np.mean(recall_scores):.4f}")
print(f"Average F1-score: {np.mean(f1_scores):.4f}")

randomforest

### Random undersampling — Random Forest

In [ ]:
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

X = df[SELECTED_FEATURES]
y = df['Accident_severity']

for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

    # Initialize Model
    rf_model = RandomForestClassifier(random_state=42)
    rf_model.fit(X_train_fold, y_train_fold)

    # Predict
    y_pred = rf_model.predict(X_test_fold)

    # Collect Metrics
    accuracy_scores.append(accuracy_score(y_test_fold, y_pred))
    precision_scores.append(precision_score(y_test_fold, y_pred, average='weighted', zero_division=0))
    recall_scores.append(recall_score(y_test_fold, y_pred, average='weighted', zero_division=0))
    f1_scores.append(f1_score(y_test_fold, y_pred, average='weighted', zero_division=0))

    print(f"Fold {fold + 1} completed")

print("\n--- Random Forest Average Results ---")
print(f"Avg Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Avg F1-score: {np.mean(f1_scores):.4f}")

XGBoost

### Random undersampling — XGBoost

In [ ]:
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

X = df[SELECTED_FEATURES]
y = df['Accident_severity']


# Loop through each fold
for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    print(f"Fold {fold + 1}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Apply RandomUnderSampler to the training data of the current fold
    rus = RandomUnderSampler(random_state=42)
    X_train_fold, y_train_fold = rus.fit_resample(X_train, y_train)

    # Initialize and train your model (e.g., RandomForestClassifier)
    model = xgb.XGBClassifier() # Or any other classifier
    model.fit(X_train_fold, y_train_fold)

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Store metrics
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    print('classification report:', classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")

# Calculate and print the average metrics across all folds
print("\nAverage Metrics across all folds:")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Average Precision: {np.mean(precision_scores):.4f}")
print(f"Average Recall: {np.mean(recall_scores):.4f}")
print(f"Average F1-score: {np.mean(f1_scores):.4f}")

#**Smote**

#k=10

## 9. SMOTE experiments

This section applies SMOTE to the training data within each validation fold.

In [ ]:
# prompt: K-Fold
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

# Define the number of folds
n_splits = 10  # You can change this value

# Initialize StratifiedKFold
kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

DecisionTree

### SMOTE — Decision Tree

In [ ]:
# Lists to store evaluation metrics for each fold
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

X = df[SELECTED_FEATURES]
y = df['Accident_severity']
# Loop through each fold
for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    print(f"Fold {fold + 1}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Apply SMOTE to the training data of the current fold
    smote = SMOTE(random_state=42)
    X_train_fold, y_train_fold = smote.fit_resample(X_train, y_train)

    # Initialize and train your model (e.g., RandomForestClassifier)
    model = DecisionTreeClassifier() # Or any other classifier
    model.fit(X_train_fold, y_train_fold)

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Store metrics
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    print('classification report:', classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")

# Calculate and print the average metrics across all folds
print("\nAverage Metrics across all folds:")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Average Precision: {np.mean(precision_scores):.4f}")
print(f"Average Recall: {np.mean(recall_scores):.4f}")
print(f"Average F1-score: {np.mean(f1_scores):.4f}")

RandomForest

### SMOTE — Random Forest

In [ ]:
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

X = df[SELECTED_FEATURES]
y = df['Accident_severity']

# Loop through each fold
for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    print(f"Fold {fold + 1}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Apply SMOTE to the training data of the current fold
    smote = SMOTE(random_state=42)
    X_train_fold, y_train_fold = smote.fit_resample(X_train, y_train)

    # Initialize and train your model (e.g., RandomForestClassifier)
    model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,              # Limit tree depth to reduce overfitting to the Slight class.
    class_weight='balanced_subsample', # Balance class weights during sampling.
    random_state=42)
    model.fit(X_train_fold, y_train_fold)
    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Store metrics
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    print('classification report:', classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")

# Calculate and print the average metrics across all folds
print("\nAverage Metrics across all folds:")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Average Precision: {np.mean(precision_scores):.4f}")
print(f"Average Recall: {np.mean(recall_scores):.4f}")
print(f"Average F1-score: {np.mean(f1_scores):.4f}")

XGB

### SMOTE — XGBoost

In [ ]:
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

X = df[SELECTED_FEATURES]
y = df['Accident_severity']


# Loop through each fold
for fold, (train_index, test_index) in enumerate(kf.split(X, y)):
    print(f"Fold {fold + 1}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Apply SMOTE to the training data of the current fold
    smote = SMOTE(random_state=42)
    X_train_fold, y_train_fold = smote.fit_resample(X_train, y_train)

    # Initialize and train your model (e.g., RandomForestClassifier)
    model = xgb.XGBClassifier() # Or any other classifier
    model.fit(X_train_fold, y_train_fold)

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Store metrics
    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    print('classification report:', classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-score: {f1:.4f}")

# Calculate and print the average metrics across all folds
print("\nAverage Metrics across all folds:")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Average Precision: {np.mean(precision_scores):.4f}")
print(f"Average Recall: {np.mean(recall_scores):.4f}")
print(f"Average F1-score: {np.mean(f1_scores):.4f}")

## 10. Final model comparison

The candidate models use the same selected features. Each model is evaluated with no sampling, random oversampling, random undersampling, and SMOTE using stratified 10-fold cross-validation on the training data. Twenty percent is held back for final testing. The model-and-sampling combination with the highest mean macro F1-score is selected because it gives every accident-severity class equal importance.

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb

# The same selected features are used for every candidate model.
final_X = df[SELECTED_FEATURES]
final_y = df['Accident_severity']

# Keep 20% unseen until the final evaluation.
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    final_X, final_y, test_size=0.2, random_state=42, stratify=final_y
)

candidate_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, random_state=42
    ),
    'XGBoost': xgb.XGBClassifier(
        objective='multi:softprob', eval_metric='mlogloss', random_state=42
    ),
}

samplers = {
    'No sampling': None,
    'Random oversampling': RandomOverSampler(random_state=42),
    'Random undersampling': RandomUnderSampler(random_state=42),
    'SMOTE': SMOTE(random_state=42),
}

inner_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
model_results = []

for sampler_name, sampler in samplers.items():
    for model_name, candidate in candidate_models.items():
        fold_scores = []

        for train_index, validation_index in inner_cv.split(X_train_final, y_train_final):
            X_train_fold = X_train_final.iloc[train_index]
            y_train_fold = y_train_final.iloc[train_index]
            X_validation = X_train_final.iloc[validation_index]
            y_validation = y_train_final.iloc[validation_index]

            if sampler is not None:
                X_train_fold, y_train_fold = clone(sampler).fit_resample(
                    X_train_fold, y_train_fold
                )

            fold_model = clone(candidate)
            fold_model.fit(X_train_fold, y_train_fold)
            validation_predictions = fold_model.predict(X_validation)
            fold_scores.append({
                'Accuracy': accuracy_score(y_validation, validation_predictions),
                'Weighted F1-score': f1_score(
                    y_validation, validation_predictions, average='weighted'
                ),
                'Macro F1-score': f1_score(
                    y_validation, validation_predictions, average='macro'
                ),
            })

        mean_scores = pd.DataFrame(fold_scores).mean()
        model_results.append({
            'Model': model_name,
            'Sampling method': sampler_name,
            'Mean Accuracy': mean_scores['Accuracy'],
            'Mean Weighted F1-score': mean_scores['Weighted F1-score'],
            'Mean Macro F1-score': mean_scores['Macro F1-score'],
        })

model_results = pd.DataFrame(model_results).sort_values(
    'Mean Macro F1-score', ascending=False
).reset_index(drop=True)
selected_model_name = model_results.loc[0, 'Model']
selected_sampler_name = model_results.loc[0, 'Sampling method']
selected_model = clone(candidate_models[selected_model_name])
selected_sampler = samplers[selected_sampler_name]

X_train_selected = X_train_final
y_train_selected = y_train_final

if selected_sampler is not None:
    X_train_selected, y_train_selected = clone(selected_sampler).fit_resample(
        X_train_final, y_train_final
    )

selected_model.fit(X_train_selected, y_train_selected)

print(f'Selected model: {selected_model_name}')
print(f'Selected sampling method: {selected_sampler_name}')
model_results

## 11. Hyperparameter tuning for XGBoost + SMOTE

XGBoost with SMOTE achieved the best macro F1-score in the model comparison. This section searches for a better combination of XGBoost settings using stratified 10-fold cross-validation on the training data. Macro F1-score is used because every severity class is equally important.

In [ ]:
from imblearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV

tuning_pipeline = Pipeline([
    ('sampler', SMOTE(random_state=42)),
    ('model', xgb.XGBClassifier(
        objective='multi:softprob', eval_metric='mlogloss',
        random_state=42, n_jobs=1
    )),
])

parameter_distributions = {
    'sampler__k_neighbors': [3, 5, 7],
    'model__n_estimators': [200, 300, 500],
    'model__max_depth': [3, 4, 5, 6],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__min_child_weight': [1, 3, 5],
    'model__subsample': [0.7, 0.85, 1.0],
    'model__colsample_bytree': [0.7, 0.85, 1.0],
    'model__gamma': [0, 0.1, 0.3],
    'model__reg_lambda': [1, 5, 10],
}

tuning_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
tuner = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=parameter_distributions,
    n_iter=12,
    scoring='f1_macro',
    cv=tuning_cv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
tuner.fit(X_train_final, y_train_final)

tuned_model = tuner.best_estimator_
print(f'Best cross-validation Macro F1-score: {tuner.best_score_:.4f}')
print('Best parameters:')
print(tuner.best_params_)

## 12. Final evaluation of the tuned model

The tuned model is evaluated once on the 20% test set that was not used during model selection or tuning. This gives an unbiased estimate of final performance.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

tuned_predictions = tuned_model.predict(X_test_final)
print(classification_report(y_test_final, tuned_predictions, zero_division=0))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_final, tuned_predictions, ax=ax, cmap='Blues'
)
ax.set_title('Confusion Matrix: Tuned XGBoost with SMOTE')
plt.show()

print(f'Final test Macro F1-score: {f1_score(y_test_final, tuned_predictions, average="macro"):.4f}')

## 13. Feature importance of the tuned model

Feature importance ranks the selected variables by their contribution to the tuned XGBoost model. The scores explain how the model makes predictions; they do not prove that a variable causes accident severity.

In [ ]:
feature_importance_df = pd.DataFrame({
    'Feature': final_X.columns,
    'Importance': tuned_model.named_steps['model'].feature_importances_
}).sort_values('Importance', ascending=False)

top10_features = feature_importance_df.head(10)

plt.figure(figsize=(8, 5))
plt.barh(top10_features['Feature'][::-1], top10_features['Importance'][::-1])
plt.xlabel('Feature importance')
plt.title('Top 10 Feature Importances: Tuned XGBoost with SMOTE')
plt.show()

top10_features